# C7 preview: different policies and actions per application
One NeMo Guardrails server, several configurations selected per request (`config_id`), no redeploy:
* **internal chatbot** (`policy-chatbot`): MASK all identifiers
* **ID-card OCR** (`policy-idocr`): NIK ALLOWED, other identifiers masked
* **block** (`policy-pii-detect`) and **log only** (`policy-pii-log`)

In [ ]:
import os, sys, json
os.chdir(os.path.expanduser("~/aigov"))           # notebooks/ and data/ live here
sys.path.insert(0, "notebooks")
import harness as h
print("Guardrails:", os.environ["NEMO_URL"])

In [ ]:
p = "Customer NIK 3273550791000456, phone 081234567890, email dewi.l@example.com: please validate."
for cfg in ["policy-chatbot", "policy-idocr"]:
    print(f"{cfg:16s} mask ->", h.transform(p, config_id=cfg)["text_out"])
print("policy-pii-detect block ->", h.check(p, config_id="policy-pii-detect")["blocked"])
print("policy-pii-log    log   ->", "allowed =", not h.check(p, config_id="policy-pii-log")["blocked"], "(event written to the guardrail log, without raw values)")

**Reading the output:** the same prompt, sent to one guardrail server with four different configurations (chosen per request, no redeploy):

* `policy-chatbot … mask`: the internal chatbot masks **all** identifiers (`[NIK]`, `[PHONE]`, `[EMAIL]`).
* `policy-idocr … mask`: the ID-card OCR app keeps the **NIK** (it needs it to validate the card) but still masks the phone and email.
* `policy-pii-detect block -> True`: a strict policy that refuses any prompt with personal data.
* `policy-pii-log log -> allowed = True`: lets the prompt through but writes an event to the guardrail log (types only, not the values), for monitoring without blocking.